# 🎬 VAJRA v1.1 — AI Media Studio

**6 tabs:** Edit & Relip · Text→Image (RealVisXL) · Face Swap · Text→Video (LTX text-only, or Wan2.2-I2V motion video with a reference photo) · Image Edit (FLUX-Kontext) · Media Studio.

**How to run:** `Runtime → Change runtime type → GPU` (**A100** recommended), then run the cells **top to bottom**.

- First run is slow (builds isolated venvs + downloads several GB of weights). Set **`USE_DRIVE = True`** in Step 2 to cache them and skip re-downloading next session.
- **No tokens needed** for the defaults (SDXL Realistic, face swap, relip, LTX). **Only** the *Image Edit* tab needs a Hugging Face token — see Step 4. Wan2.2-I2V (Text→Video's photo-animation mode) may also need one depending on its access settings -- check huggingface.co/Wan-AI/Wan2.2-I2V-A14B-Diffusers if it fails to load.
- Cells **7b / 7c / 7d** are optional/heavy — skip unless you want LTX video, MuseTalk, or Wan2.2-I2V motion video.

## ✅ **Step 1 — Check the GPU**
Confirms an A100/L4 is attached. If this errors, set `Runtime → Change runtime type → GPU` first.

In [ ]:
# 1. Check GPU
!nvidia-smi

## 💾 **Step 2 — (Optional) Cache model weights on Drive**
Set `USE_DRIVE = True` to cache the multi-GB **model weights** on your Drive, so cell 7 skips re-downloading them next session. **Venvs stay local** — Google Drive can't run a venv's executables, so cell 6 always rebuilds them.

In [ ]:
# 2. (Optional) cache MODEL WEIGHTS on Drive to skip re-downloading next session.
#    NOTE: venvs are deliberately NOT put on Drive — Google Drive can't execute a
#    venv's python ("bad interpreter: Permission denied"), so they always build
#    locally on /content. Only the (large) weights are cached.
USE_DRIVE = False  # set True to cache model weights on your Drive
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['IMAGE_TALK_MODELS'] = '/content/drive/MyDrive/image_talk/models'
    os.makedirs(os.environ['IMAGE_TALK_MODELS'], exist_ok=True)
    os.environ.pop('IMAGE_TALK_VENVS', None)   # keep venvs local (Drive can't run them)

## 📥 **Step 3 — Get the project**
Clones the repo from GitHub into `/content`. Public repo works as-is; for a private repo add a `GITHUB_TOKEN` Colab secret (🔑 icon).

In [ ]:
# 3. Get the project from GitHub (public repo works as-is).
import os
REPO_USER = 'raunakmalix-ctrl'
REPO_NAME = 'VAJRA-v1.1'
ROOT = f'/content/{REPO_NAME}'

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass

auth = f'{token}@' if token else ''
REPO_URL = f'https://{auth}github.com/{REPO_USER}/{REPO_NAME}.git'
if not os.path.isdir(ROOT):
    !git clone $REPO_URL $ROOT
os.environ['IMAGE_TALK_ROOT'] = ROOT
%cd $ROOT

## 🔑 **Step 4 — Hugging Face token (only for the Image Edit tab)**

The defaults need **no token**. You only need one for the **Image Edit** tab (FLUX-Kontext), or to use FLUX in Text→Image.

**Get it before running anything heavy:**
1. Free account at **huggingface.co**.
2. Token: **huggingface.co/settings/tokens → New token → type *Read* → Create →** copy the `hf_…` value.
3. **Accept the license** (logged in) on the model page you'll use:
   - Image Edit → **huggingface.co/black-forest-labs/FLUX.1-Kontext-dev** → click **Agree**.
   - (Optional) FLUX text→image → huggingface.co/black-forest-labs/FLUX.1-dev.
4. Paste the token into the cell below. Access is usually granted instantly.

*Don't need image editing? Leave it blank and use **SDXL Realistic** (the default).*

In [ ]:
# 4. Hugging Face token — ONLY needed for the gated FLUX models (Image Edit / FLUX).
#    Leave blank to use SDXL Realistic (default, no token).
import os
os.environ['HF_TOKEN'] = ''  # <- paste your hf_... token here, or leave empty

## ⚙️ **Step 5 — Main environment**
Installs ffmpeg, clones the model repos (Wav2Lip, LatentSync, MuseTalk, CodeFormer, …), and pip-installs the main env. A few minutes. **Re-run this after a `git pull` that adds new repos.**

In [ ]:
# 5. Main env: ffmpeg + clone model repos + pip install (a few minutes)
!bash setup/install_main.sh

## 🧩 **Step 6 — Build the isolated venvs**
Creates separate Python envs for XTTS / LatentSync (their dependencies conflict with the main env). **Slow on first run.**

In [ ]:
# 6. Build the isolated venvs (XTTS / LatentSync). Slow first run.
!bash setup/make_venvs.sh

## ⬇️ **Step 7 — Download model weights**
Fetches the open weights (XTTS, LatentSync, GFPGAN, inswapper, …). Several GB. Idempotent — safe to re-run.

In [ ]:
# 7. Download model weights
!python setup/download_models.py

## 🎥 **Step 7b — (Optional) LTX video venv**
Only if you want the **Text→Video** tab. Builds a separate Python 3.12 venv. Heavy — skip otherwise.

In [ ]:
# 7b. (Optional) LTX-Video 0.9.7-distilled text -> video (open, no token).
!bash setup/make_ltx_venv.sh

## 👄 **Step 7c — (Optional) MuseTalk venv**
Only if you want the **MuseTalk** sharper-lips option (Edit & Relip). Heavy mmlab stack — skip otherwise.

In [ ]:
# 7c. (Optional) MuseTalk enhanced lip-sync venv (mmlab stack + weights).
!bash setup/make_musetalk_venv.sh

## 🕺 **Step 7d — (Optional) Wan2.2-I2V motion-video venv**
Only if you want Text→Video's reference-photo mode (identity-preserving motion video, no audio/lip-sync -- the photo is animated by the prompt). Heavy: a ~27B parameter model, large first download. Skip otherwise.

In [ ]:
# 7d. (Optional) Wan2.2-I2V motion video -- animate a reference photo from a prompt.
!bash setup/make_wan_venv.sh

## 🚀 **Step 8 — Launch the app**
Starts the Gradio app and prints a public **`*.gradio.live`** link — click it to open all 6 tabs. Keep this cell running while you use the app.

In [ ]:
# 8. Launch the app — click the public *.gradio.live link in the output
os.environ['IMAGE_TALK_SHARE'] = '1'
!python app.py